<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/t06_MD_enhanced_martini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Enhanced sampling and simplified models in MD

In this lab experience you will go beyond standard atomistic MD in two directions:

*   **Enhanced sampling**: rare events, such as the transitions between conformations separated by high free-energy barriers, are not observed in a standard MD simulation. With **metadynamics** we add a history-dependent bias potential that pushes the system away from the states it has already visited, and we learn the free-energy landscape along a few chosen collective variables (CVs). The test system is alanine dipeptide in vacuum, that you already know from the lectures.
*   **Simplified (coarse-grained) models**: with the **Martini** force field groups of about four heavy atoms are represented by a single bead. Much larger systems and longer times become accessible. We use it to study the self-assembly and the physical properties of lipid bilayers.

The reference lectures are:

*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/02_StochasticMolecules.pdf
*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/05_MolecularDynamics.pdf
*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/06_EnhancedMD.pdf
*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/08_QM_MM_more.pdf

At odds with the other tasks, here instructions and reports are coupled: **GROUP 1** follows the part on **enhanced sampling** (Part A) and **GROUP 2** the part on **Martini simulations** (Part B). Everybody runs the setup cells. You are of course welcome to look at (and run) the other part too.

## Preliminary setup

The simulations of Part A are tiny and run well on the CPU. Those of Part B are faster on a GPU, so if you are in Group 2 check that you have one:

In [ ]:
!nvidia-smi || echo "no NVIDIA GPU found"

If the above command is not found, select `Runtime -> Change runtime type` and choose T4 GPU, then run the cell again.

### Installing GROMACS and PLUMED

We install GROMACS (2026.3, with GPU support) and [PLUMED](https://www.plumed.org) (2.10) from [conda-forge](https://conda-forge.org) using `micromamba`, as in T04. PLUMED is a library to compute collective variables and to perform enhanced sampling simulations with many MD codes: GROMACS loads it when `mdrun` is run with the `-plumed` option. About 2.5 GB are downloaded, so the installation takes a few minutes.

Colab forgets everything you installed when the session ends: **run this cell again every time you start a new session.**

If you run the notebook on your own computer with the course conda environment (see `Notebooks/README.md`), both programs are already installed and the cell only prints their versions.

In [ ]:
%%bash
# install GROMACS 2026.3 with CUDA (GPU) support and PLUMED 2.10 from conda-forge, unless they are already installed
if ! command -v gmx > /dev/null || ! command -v plumed > /dev/null; then
  cd /content
  curl -sL -o micromamba https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64
  chmod +x micromamba
  ./micromamba create -y -q -r /content/mamba -p /content/gmx -c conda-forge "gromacs=2026.3=nompi_cuda*" "plumed=2.10"
  ln -sf /content/gmx/bin/gmx /usr/local/bin/gmx
  ln -sf /content/gmx/bin/plumed /usr/local/bin/plumed
fi
gmx --version | grep -E "GROMACS version|GPU support|Plumed support"
echo "PLUMED version: $(plumed info --version)"

GROMACS finds the PLUMED library through the environment variable `PLUMED_KERNEL`, which we set here. We also install `MDAnalysis` (analysis of the trajectories) and `py3Dmol` (visualisation in the notebook).

In [ ]:
import os, glob, shutil
# PLUMED_KERNEL is the path of the PLUMED library, in the lib folder next to the bin folder of the plumed program
plumed_bin = os.path.dirname(os.path.realpath(shutil.which('plumed')))
os.environ['PLUMED_KERNEL'] = os.path.abspath(glob.glob(os.path.join(plumed_bin, '..', 'lib', 'libplumedKernel.*'))[0])
print('PLUMED_KERNEL =', os.environ['PLUMED_KERNEL'])

%pip install -q MDAnalysis py3Dmol

We work in a folder of the virtual machine (set `use_drive = True` to work in your Google Drive instead: this is safer for the long runs, because the files survive a disconnection of Colab, and it is needed to use the results in the report). The cell can be run more than once.

In [ ]:
import os
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()

use_drive = False        # True: keep your files on Google Drive (Colab will ask your permission)
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    local_path = '/content/drive/MyDrive/Structural_Bioinformatics/T06'
else:
    local_path = os.path.join(BASE_DIR, 'T06')
os.makedirs(local_path, exist_ok=True)
%cd $local_path

In [ ]:
# the input files of the simulations are in the Data folder of the course repository:
# if this notebook is run from a copy of the repository we use it, otherwise we download the repository
if os.path.isdir(os.path.join(BASE_DIR, '..', 'Data', 'md', 'metad')):
    REPO_DIR = os.path.abspath(os.path.join(BASE_DIR, '..'))
else:
    REPO_DIR = os.path.join(BASE_DIR, 'sb')
    if os.path.isdir(REPO_DIR):
        !cd "$REPO_DIR" && git pull -q
    else:
        !git clone -q --depth 1 https://github.com/carlocamilloni/Structural-Bioinformatics.git "$REPO_DIR"
METAD_DIR = os.path.join(REPO_DIR, 'Data', 'md', 'metad')        # Part A
MARTINI_DIR = os.path.join(REPO_DIR, 'Data', 'martini')    # Part B
print('input files from', METAD_DIR, 'and', MARTINI_DIR)

Two small functions to read the text files written by GROMACS (`.xvg`) and by PLUMED (`COLVAR`, `HILLS`, free-energy files). In PLUMED files the first line starts with `#! FIELDS` and gives the names of the columns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def read_xvg(filename):
    """Read a GROMACS .xvg file.
    Returns (data, legends): data is a 2D numpy array whose first column is the x axis,
    legends is the list of names of the other columns (it can be empty)."""
    rows, legends = [], []
    with open(filename) as f:
        for line in f:
            if line.startswith('@') and 'legend "' in line and line.split()[1].startswith('s'):
                legends.append(line.split('"')[1])
            elif not line.startswith(('#', '@')) and line.strip():
                rows.append([float(x) for x in line.split()])
    return np.array(rows), legends

def read_plumed(filename):
    """Read a PLUMED output file (COLVAR, HILLS, fes). Returns a dictionary {column name: numpy array}."""
    with open(filename) as f:
        fields = f.readline().split()[2:]          # first line: #! FIELDS name1 name2 ...
    data = np.loadtxt(filename, comments='#', ndmin=2)
    return {name: data[:, i] for i, name in enumerate(fields)}

# Part A: Metadynamics of alanine dipeptide (GROUP 1)

Alanine dipeptide (Ace-Ala-NMe) is the simplest model of a protein backbone: its conformations are described by the two dihedral angles $\phi$ and $\psi$ of the alanine residue. In vacuum it has two main groups of states: for $\phi<0$ the extended/β region (also called C7eq) and the right-handed α-helical region (αR); for $\phi>0$ the left-handed region (αL, also called C7ax). The transition between $\phi<0$ and $\phi>0$ is slow: it is a nice example of a rare event that we can make frequent with enhanced sampling.

Free energy and probability are related by the Boltzmann distribution, $F(s) = -k_BT \ln P(s) + \mathrm{const}$, where $s$ is any collective variable (here $\phi$ and $\psi$) and $k_BT = 2.494$ kJ/mol at 300 K. The population of a state $A$ is the integral of the probability over the region of the state, $P_A = \int_A e^{-F(s)/k_BT} ds \, / \int e^{-F(s)/k_BT} ds$.

In [ ]:
kT = 0.0083144626 * 300      # kJ/mol at 300 K
metad_path = os.path.join(local_path, 'MTD')
os.makedirs(metad_path, exist_ok=True)
%cd $metad_path

## System preparation

We build the topology with the AMBER99SB-ILDN force field, without water, and place the molecule in a box with 1.2 nm between the molecule and the box sides. Since there is no solvent, the interactions are computed with a simple cut-off of 1.0 nm. Then we minimise the energy.

In [ ]:
%cp $REPO_DIR/Data/md/inputs/aladp.pdb .
%cp $METAD_DIR/em-vacuum.mdp $METAD_DIR/md-vacuum.mdp .
!gmx pdb2gmx -f aladp.pdb -o aladp.gro -water none -ff amber99sb-ildn
!gmx editconf -f aladp.gro -bt cubic -d 1.2 -o boxed.gro
!gmx grompp -f em-vacuum.mdp -c boxed.gro -p topol.top -o em.tpr
!gmx mdrun -deffnm em

In [ ]:
# from the energy minimised structure we also make a reference PDB file: PLUMED reads from it the names of the atoms
!gmx editconf -f em.gro -o reference.pdb

Now a 10 ns long standard MD simulation at 300 K. For such a small molecule a GPU would not help, so we ask GROMACS to use the CPU (`-nb cpu`) with a single thread: it should take a couple of minutes.

In [ ]:
!gmx grompp -f md-vacuum.mdp -c em.gro -p topol.top -o md.tpr
!gmx mdrun -deffnm md -nb cpu -ntmpi 1 -ntomp 1 -v

## Analysis of a standard MD simulation with PLUMED

PLUMED can be used both to analyse trajectories, computing collective variables, and to bias simulations with them. You can learn more about its syntax and features in the [PLUMED tutorials](https://www.plumed-tutorials.org) and find the inputs of many published simulations in [PLUMED-NEST](https://www.plumed-nest.org).

A PLUMED input is a text file where each line defines an *action*: a quantity to compute (e.g. `TORSION`), a bias to apply (e.g. `METAD`) or an operation such as printing (`PRINT`). Let's look at a first input that computes $\phi$ and $\psi$:

In [ ]:
%cp $METAD_DIR/plumed-*.dat .
!cat plumed-cv.dat

`@phi-2` and `@psi-2` are shortcuts, defined by `MOLINFO`, for the four atoms that define the dihedral angles $\phi$ and $\psi$ of residue 2 (alanine). The `plumed driver` program applies the input to each frame of a trajectory:

In [ ]:
!plumed driver --plumed plumed-cv.dat --mf_xtc md.xtc
!head COLVAR-CV.txt

The file starts with some header lines: the names of the columns (time, phi and psi) and the information that both angles are periodic between $-\pi$ and $\pi$. Then the data: angles are in radians, time in ps.

In [ ]:
cv = read_plumed('COLVAR-CV.txt')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(cv['time'] / 1000, cv['phi'], '.', ms=1)
axes[0].set_ylabel('$\\phi$ (rad)')
axes[1].plot(cv['time'] / 1000, cv['psi'], '.', ms=1)
axes[1].set_ylabel('$\\psi$ (rad)')
for ax in axes[:2]:
    ax.set_xlabel('Time (ns)')
axes[2].plot(cv['phi'], cv['psi'], '.', ms=1)
axes[2].set_xlabel('$\\phi$ (rad)')
axes[2].set_ylabel('$\\psi$ (rad)')
axes[2].set_xlim(-np.pi, np.pi)
axes[2].set_ylim(-np.pi, np.pi)
plt.show()

From a standard simulation the free energy can be estimated directly from the histogram of the CVs, $F(s) = -k_BT \ln P(s)$. The functions below compute it in 1D and 2D, optionally with a weight for each frame (we will need weights later). Regions never visited have $P=0$ and an infinite free energy: they appear empty in the plots.

In [ ]:
def fes_1d(x, weights=None, bins=72):
    """Free energy (kJ/mol, minimum set to 0) from the histogram of a periodic variable between -pi and pi."""
    hist, edges = np.histogram(x, bins=bins, range=(-np.pi, np.pi), weights=weights, density=True)
    centers = 0.5 * (edges[1:] + edges[:-1])
    with np.errstate(divide='ignore'):
        fes = -kT * np.log(hist)
    return centers, fes - fes.min()

def fes_2d(x, y, weights=None, bins=72):
    """Free energy (kJ/mol, minimum 0) from the 2D histogram of two periodic variables. fes[i, j] is at (x[i], y[j])."""
    hist, xe, ye = np.histogram2d(x, y, bins=bins, range=[[-np.pi, np.pi], [-np.pi, np.pi]], weights=weights, density=True)
    with np.errstate(divide='ignore'):
        fes = -kT * np.log(hist)
    return 0.5 * (xe[1:] + xe[:-1]), 0.5 * (ye[1:] + ye[:-1]), fes - fes.min()

def plot_fes_2d(x, y, fes, title='', fmax=50, ax=None):
    """Contour plot of a 2D free energy, with fes[i, j] at (x[i], y[j])."""
    ax = ax or plt.gca()
    levels = np.arange(0, fmax + 1, 2.5)
    c = ax.contourf(x, y, np.clip(fes, 0, fmax).T, levels=levels, cmap='viridis')
    plt.colorbar(c, ax=ax, label='Free energy (kJ/mol)')
    ax.set_xlabel('$\\phi$ (rad)')
    ax.set_ylabel('$\\psi$ (rad)')
    ax.set_title(title)

phi_md, f_phi_md = fes_1d(cv['phi'])
psi_md, f_psi_md = fes_1d(cv['psi'])
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].plot(phi_md, f_phi_md)
axes[0].set_xlabel('$\\phi$ (rad)')
axes[1].plot(psi_md, f_psi_md)
axes[1].set_xlabel('$\\psi$ (rad)')
for ax in axes[:2]:
    ax.set_ylabel('Free energy (kJ/mol)')
plot_fes_2d(*fes_2d(cv['phi'], cv['psi']), title='standard MD', ax=axes[2])
plt.show()

We define the states as regions of the $(\phi, \psi)$ plane and compute their populations as the fraction of frames in each region:

*   β (C7eq): $\phi<0$ and $\psi>1$ rad or $\psi<-2$ rad
*   αR: $\phi<0$ and $-2<\psi<1$ rad
*   αL (C7ax): $\phi>0$

The same function works with weighted frames (we will need it later).

In [ ]:
def states(phi, psi):
    """Dictionary {state name: boolean array} telling which frames are in each state."""
    return {'beta': (phi < 0) & ((psi > 1) | (psi < -2)),
            'alphaR': (phi < 0) & (psi > -2) & (psi < 1),
            'alphaL': phi > 0}

def populations(phi, psi, weights=None):
    weights = np.ones(len(phi)) if weights is None else weights
    return {name: weights[inside].sum() / weights.sum() for name, inside in states(phi, psi).items()}

for name, p in populations(cv['phi'], cv['psi']).items():
    print(f'{name:8s} {p:6.3f}')

**Questions:** How many times does the system go from $\phi<0$ to $\phi>0$ in 10 ns? Can you trust the free-energy difference between αL and the other two states? And that between β and αR? Try a longer simulation (change `nsteps` in `md-vacuum.mdp` and run `grompp` and `mdrun` again with a different name): how long should it be to estimate the population of αL?

## Well-tempered metadynamics on $\phi$ and $\psi$

Now we run the same simulation biasing $\phi$ and $\psi$ with well-tempered metadynamics. Let's look at the input file:

In [ ]:
!cat plumed-mtd.dat

The `METAD` action takes as arguments the two dihedral angles and sets some parameters:

*   `PACE` is how often (in time steps) a new Gaussian is added to the bias: it is the pace at which we learn from the simulation.
*   `HEIGHT` and `SIGMA` are the initial height (kJ/mol) and the widths (rad) of the Gaussians. `SIGMA` should be a fraction of the fluctuations of the CVs in a single state.
*   `BIASFACTOR` $\gamma$ makes the metadynamics *well-tempered*: the height of the new Gaussians decreases where the bias is already large, so that the bias converges. In the end the CVs are sampled from a smoothed distribution $P(s)^{1/\gamma}$: with $\gamma=10$ a barrier of 100 kJ/mol becomes a barrier of 10 kJ/mol. The final bias is related to the free energy by $V(s) = -(1-1/\gamma) F(s) + \mathrm{const}$.
*   `GRID_MIN`, `GRID_MAX`: the bias is stored on a grid, which makes it much faster to compute.
*   `FILE`: the Gaussians are written in the HILLS file, from which the free energy is reconstructed.
*   `CALC_RCT`: computes the quantity $c(t)$ needed to reweight the simulation (see below).

The `PRINT` line writes the CVs, the bias and the reweighting bias `metad.rbias` $=V(s,t)-c(t)$ every 500 steps. We start from the same input (`md.tpr`): the `-plumed` flag tells GROMACS to activate PLUMED with our input file.

In [ ]:
# run it, expected to last a few minutes
!gmx mdrun -s md.tpr -deffnm mtd -plumed plumed-mtd.dat -nb cpu -ntmpi 1 -ntomp 1 -v

### Analysis

During the simulation PLUMED computed on-the-fly the dihedral angles and the bias, and wrote them in `COLVAR-MTD.txt`:

In [ ]:
!head -5 COLVAR-MTD.txt
mtd = read_plumed('COLVAR-MTD.txt')

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
axes[0, 0].plot(mtd['time'] / 1000, mtd['phi'], '.', ms=2)
axes[0, 0].set_ylabel('$\\phi$ (rad)')
axes[0, 1].plot(mtd['time'] / 1000, mtd['psi'], '.', ms=2)
axes[0, 1].set_ylabel('$\\psi$ (rad)')
axes[1, 0].plot(mtd['phi'], mtd['psi'], '.', ms=2)
axes[1, 0].set_xlabel('$\\phi$ (rad)')
axes[1, 0].set_ylabel('$\\psi$ (rad)')
axes[1, 1].plot(mtd['time'] / 1000, mtd['metad.bias'], '.', ms=2)
axes[1, 1].set_ylabel('Bias (kJ/mol)')
for ax in (axes[0, 0], axes[0, 1], axes[1, 1]):
    ax.set_xlabel('Time (ns)')
plt.show()

The speed-up of the sampling should be evident, but it is obtained adding energy to the system (the bias): the distribution of the sampled configurations is not the Boltzmann distribution of the original system, so we cannot simply count frames as we did before.

### Free energy from the bias

The free energy is estimated from the accumulated bias, $F(s) = -\frac{\gamma}{\gamma-1} V(s)$. PLUMED does it with `sum_hills`, which sums the Gaussians in the HILLS file. With `--stride` it writes an estimate every 1000 Gaussians (1 ns), so that we can follow its evolution; with `--idw` it integrates out the other variable to give the free energy of a single CV. The free energies are shifted to have a minimum of zero.

In [ ]:
# free energy as a function of phi: one file every 1000 Gaussians (fes-phi_0.dat, fes-phi_1.dat, ...) and one at the end
!plumed sum_hills --hills HILLS-MTD.txt --mintozero --outfile fes-phi_ --idw phi --kt {kT} --stride 1000
# free energy as a function of psi
!plumed sum_hills --hills HILLS-MTD.txt --mintozero --outfile fes-psi_ --idw psi --kt {kT} --stride 1000
# final estimate of the free energy as a function of phi and psi
!plumed sum_hills --hills HILLS-MTD.txt --mintozero --outfile fes-2D.dat
!ls fes*

In [ ]:
def fes_files(prefix, hills, stride=1000):
    """The free-energy files written by sum_hills --stride, in order of time.
    Returns a list of (time in ns, file name): the time is that of the last Gaussian included in each file."""
    times = read_plumed(hills)['time']
    files = sorted(glob.glob(f'{prefix}*.dat'), key=lambda f: int(f[len(prefix):-4]))
    result = []
    for i, f in enumerate(files):
        t = times[min((i + 1) * stride, len(times)) - 1] / 1000
        if not result or t > result[-1][0]:          # the last file can be a copy of the previous one
            result.append((t, f))
    return result

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, cv_name, x_md, f_md in [(axes[0], 'phi', phi_md, f_phi_md), (axes[1], 'psi', psi_md, f_psi_md)]:
    files = fes_files(f'fes-{cv_name}_', 'HILLS-MTD.txt')
    for i, (t, f) in enumerate(files):
        fes = read_plumed(f)
        ax.plot(fes[cv_name], fes['projection'], color=plt.cm.viridis(i / len(files)), label=f'{t:.0f} ns')
    ax.plot(x_md, f_md, 'k--', label='standard MD')
    ax.set_xlabel(f'$\\{cv_name}$ (rad)')
    ax.set_ylabel('Free energy (kJ/mol)')
    ax.set_ylim(0, 60)
    ax.legend(ncol=2)
plt.show()

A simple and useful test of convergence is to follow the free-energy difference between two regions as a function of time: when the simulation is converged, the difference stops changing (it fluctuates around a constant value). The free-energy difference between $\phi>0$ and $\phi<0$ is $\Delta F = -k_BT \ln (P_{\phi>0}/P_{\phi<0})$, where each probability is the integral of $e^{-F/k_BT}$ over the region.

In [ ]:
def delta_f(x, fes, region_a, region_b):
    """Free energy of region b minus that of region a (boolean arrays on the grid x)."""
    p = np.exp(-fes / kT)
    return -kT * np.log(p[region_b].sum() / p[region_a].sum())

times, df = [], []
for t, f in fes_files('fes-phi_', 'HILLS-MTD.txt'):
    fes = read_plumed(f)
    times.append(t)
    df.append(delta_f(fes['phi'], fes['projection'], fes['phi'] < 0, fes['phi'] > 0))
plt.plot(times, df, 'o-')
plt.xlabel('Time (ns)')
plt.ylabel('$F(\\phi>0) - F(\\phi<0)$ (kJ/mol)')
plt.show()

Finally the 2D free-energy surface. `sum_hills` writes it on a grid, one line for each point: we reshape it into a 2D array. Compare it with the one from the standard simulation.

In [ ]:
def read_fes_2d(filename, cv_1='phi', cv_2='psi'):
    """Read a 2D free energy written by sum_hills. Returns x, y, fes with fes[i, j] at (x[i], y[j])."""
    d = read_plumed(filename)
    x, y = np.unique(d[cv_1]), np.unique(d[cv_2])
    fes = d['file.free'].reshape(len(y), len(x)).T      # sum_hills writes the first variable fastest
    return x, y, fes

phi_grid, psi_grid, fes_mtd = read_fes_2d('fes-2D.dat')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_fes_2d(phi_grid, psi_grid, fes_mtd, 'metadynamics', ax=axes[0])
plot_fes_2d(*fes_2d(cv['phi'], cv['psi']), title='standard MD', ax=axes[1])
plt.show()

The populations of the states are the integrals of $e^{-F/k_BT}$ over their regions:

In [ ]:
PHI, PSI = np.meshgrid(phi_grid, psi_grid, indexing='ij')     # the (phi, psi) coordinates of each point of the grid
p = np.exp(-fes_mtd / kT)
for name, inside in states(PHI, PSI).items():
    print(f'{name:8s} {p[inside].sum() / p.sum():6.3f}')

**Questions:** Compare the free energies and the populations with those of the standard MD. Which ones agree? Which ones are different and why? Is the metadynamics simulation converged after 10 ns? What happens to the free energy of the regions that have never been visited?

## The choice of the collective variables

Metadynamics works if the CVs include the slow degrees of freedom of the system. Here we know that the slow motion is the change of $\phi$, while $\psi$ is fast. What happens if we bias only one of the two angles? We run two more simulations, each biasing only one CV. Look at the input files: they are the same as before, but with one argument only. Both runs print $\phi$ and $\psi$.

In [ ]:
!cat plumed-mtd-psi.dat
!gmx mdrun -s md.tpr -deffnm mtd-psi -plumed plumed-mtd-psi.dat -nb cpu -ntmpi 1 -ntomp 1
!gmx mdrun -s md.tpr -deffnm mtd-phi -plumed plumed-mtd-phi.dat -nb cpu -ntmpi 1 -ntomp 1

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for row, biased in enumerate(['psi', 'phi']):
    run = read_plumed(f'COLVAR-{biased}.txt')
    axes[row, 0].plot(run['time'] / 1000, run['phi'], '.', ms=2, label='$\\phi$')
    axes[row, 0].plot(run['time'] / 1000, run['psi'], '.', ms=2, label='$\\psi$')
    axes[row, 0].set_xlabel('Time (ns)')
    axes[row, 0].set_ylabel('Angle (rad)')
    axes[row, 0].set_title(f'bias on $\\{biased}$ only')
    axes[row, 0].legend()
    axes[row, 1].plot(run['phi'], run['psi'], '.', ms=2)
    axes[row, 1].set_xlabel('$\\phi$ (rad)')
    axes[row, 1].set_ylabel('$\\psi$ (rad)')
plt.show()

In [ ]:
# the free energy as a function of the biased CV, every ns
for biased in ['psi', 'phi']:
    !plumed sum_hills --hills HILLS-{biased}.txt --mintozero --outfile fes-only{biased}_ --kt {kT} --stride 1000
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, biased, x_ref, f_ref in [(axes[0], 'psi', psi_md, f_psi_md), (axes[1], 'phi', phi_md, f_phi_md)]:
    files = fes_files(f'fes-only{biased}_', f'HILLS-{biased}.txt')
    for i, (t, f) in enumerate(files):
        fes = read_plumed(f)
        ax.plot(fes[biased], fes['file.free'], color=plt.cm.viridis(i / len(files)), label=f'{t:.0f} ns')
    ax.set_xlabel(f'$\\{biased}$ (rad)')
    ax.set_ylabel('Free energy (kJ/mol)')
    ax.set_title(f'bias on $\\{biased}$ only')
    ax.set_ylim(0, 60)
    ax.legend(ncol=2)
plt.show()

**Questions:** In which of the two simulations does $\phi$ change often? When only $\psi$ is biased, does the free energy as a function of $\psi$ converge? Why does it change abruptly from time to time? (Hint: look at what $\phi$ is doing at that moment. This is the typical signature of a missing slow CV, known as *hysteresis*.)

## Reweighting: free energies of CVs that were not biased

With a converged metadynamics we can compute the free energy as a function of *any* quantity, not only of the biased CVs. Each frame must be counted with a weight that removes the effect of the bias. For well-tempered metadynamics, after an initial transient, the weight of a frame at time $t$ is

$$ w(t) \propto e^{\left[V(s(t),t) - c(t)\right]/k_BT}$$

where $V(s(t),t)$ is the bias felt by the frame and $c(t)$ is a time-dependent constant that takes into account that the bias is growing (Tiwary and Parrinello, 2015). $V - c(t)$ is the quantity `metad.rbias` that we printed in the COLVAR files. Here we use the run where only $\phi$ was biased to compute the free energy as a function of $\psi$, which was not biased, and the 2D free energy.

In [ ]:
run = read_plumed('COLVAR-phi.txt')
keep = run['time'] > 1000                                # discard the first ns, while the bias is growing quickly
weights = np.exp((run['metad.rbias'][keep] - run['metad.rbias'][keep].max()) / kT)   # subtract the maximum to avoid overflows

# free energy as a function of psi: reweighted run with the bias on phi vs metadynamics on phi and psi
psi_w, f_psi_w = fes_1d(run['psi'][keep], weights)
fes_2d_psi = read_plumed(fes_files('fes-psi_', 'HILLS-MTD.txt')[-1][1])
plt.figure(figsize=(8, 5))
plt.plot(psi_w, f_psi_w, label='bias on $\\phi$, reweighted')
plt.plot(fes_2d_psi['psi'], fes_2d_psi['projection'], label='bias on $\\phi$ and $\\psi$')
plt.plot(*fes_1d(run['psi'][keep]), ':', label='bias on $\\phi$, NOT reweighted')
plt.xlabel('$\\psi$ (rad)')
plt.ylabel('Free energy (kJ/mol)')
plt.ylim(0, 40)
plt.legend()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_fes_2d(*fes_2d(run['phi'][keep], run['psi'][keep], weights, bins=50), title='bias on $\\phi$, reweighted', ax=axes[0])
plot_fes_2d(phi_grid, psi_grid, fes_mtd, 'bias on $\\phi$ and $\\psi$', ax=axes[1])
plt.show()

print('populations from the reweighted run with the bias on phi:')
for name, p in populations(run['phi'][keep], run['psi'][keep], weights).items():
    print(f'{name:8s} {p:6.3f}')

**Questions:** Does the reweighted free energy agree with the one obtained biasing both angles? What happens without the weights? Which regions of the 2D surface are noisy in the reweighted estimate, and why? Why does reweighting work well here, while biasing only $\psi$ did not?

**To visualise the simulations** you can download `mtd.xtc` and `em.gro` and open them with VMD (open `em.gro`, then `File -> Load Data Into Molecule` with `mtd.xtc`) or ChimeraX (`open em.gro`, `open mtd.xtc structureModel #1`).

# Part B: Martini simulations of lipid bilayers (GROUP 2)

In the Martini model about four heavy atoms (and their hydrogens) are represented by one bead; one Martini water bead (`W`) represents four water molecules. Beads interact through Lennard-Jones and Coulomb potentials, with parameters chosen to reproduce the partitioning free energies of small molecules between water and organic solvents. The smoother interactions and the fewer particles allow a time step of 20 fs and make the simulations more than 100 times faster than atomistic ones.

We use Martini 3 (Souza et al., Nat. Methods 2021) with the recently refined lipid parameters (Pedersen et al., ACS Cent. Sci. 2025) and cholesterol (Borges-Araújo et al., JCTC 2023). Dipalmitoyl-phosphatidylcholine (DPPC) is represented by 12 beads: the choline (`NC3`) and the phosphate (`PO4`) of the head group, two glycerol beads (`GL1`, `GL2`) and four beads for each of the two palmitoyl tails (`C1A`-`C4A` and `C1B`-`C4B`).

Martini uses reaction-field electrostatics with a relative dielectric constant of 15 and cut-offs of 1.1 nm. Look at the `.mdp` files in `Data/martini` for the details. The units are the same as in atomistic GROMACS simulations (nm, ps, kJ/mol); MDAnalysis uses Å.

In [ ]:
import warnings
warnings.filterwarnings('ignore')    # hide some harmless warnings of MDAnalysis
import MDAnalysis as mda
import py3Dmol

martini_path = os.path.join(local_path, 'martini')
os.makedirs(martini_path, exist_ok=True)
%cd $martini_path
# force field (.itp), topology (.top) and parameter (.mdp) files
!cp $MARTINI_DIR/*.itp $MARTINI_DIR/*.mdp $MARTINI_DIR/*.top $MARTINI_DIR/*.gro .
!ls

Coarse-grained structures cannot be shown with the standard representations (cartoons, bonds guessed from distances...). The function below shows the beads as spheres coloured by type: choline in blue, phosphate in orange, glycerol in pink, tails in grey, cholesterol in yellow (with its hydroxyl in red). Water is not shown. It reads a few frames of a trajectory, so that you can play them.

In [ ]:
BEAD_COLORS = {'NC3': 'blue', 'PO4': 'orange', 'GL1': 'pink', 'GL2': 'pink', 'ROH': 'red'}

def show_cg(topology, trajectory=None, frames=20, selection='not resname W NA CL', width=700, height=500):
    """Show a Martini system (beads as spheres) with py3Dmol: the last frame of topology, or a few frames of the trajectory."""
    u = mda.Universe(topology, trajectory) if trajectory else mda.Universe(topology)
    atoms = u.select_atoms(selection)
    step = max(1, len(u.trajectory) // frames)
    with mda.Writer('show.pdb', multiframe=True) as w:
        for ts in u.trajectory[::step]:
            w.write(atoms)
    view = py3Dmol.view(width=width, height=height)
    view.addModelsAsFrames(open('show.pdb').read(), 'pdb')
    view.setStyle({'sphere': {'radius': 2.4, 'color': 'lightgrey'}})
    view.setStyle({'resn': 'CHOL'}, {'sphere': {'radius': 2.4, 'color': 'yellow'}})
    for name, color in BEAD_COLORS.items():
        view.setStyle({'atom': name}, {'sphere': {'radius': 2.4, 'color': color}})
    view.addUnitCell()
    view.zoomTo()
    if len(u.trajectory) > 1 and trajectory:
        view.animate({'loop': 'forward', 'interval': 200})
    view.show()

## Self-assembly of a DPPC bilayer

We begin with the self-assembly of a DPPC bilayer from a random mixture of lipids and water. The gromacs tool `insert-molecules` takes the conformation of a single DPPC (`DPPC-single.gro`) and places 128 copies in a cubic box at random positions and orientations, checking that they do not overlap. The value of `-radius` (the van der Waals radius used to check the overlaps) has been increased from its default atomistic value (0.105 nm) to reflect the size of the Martini beads.

In [ ]:
%mkdir -p selfassembly
%cd selfassembly
!cp ../*.itp .              # the force-field files must be in the same folder as the topology
!gmx insert-molecules -ci ../DPPC-single.gro -box 7.5 7.5 7.5 -nmol 128 -radius 0.21 -try 500 -o 128_noW.gro

The topology file `martini_lipids.top` includes all the force-field files we need (beads, lipids, cholesterol, water and ions); we add to it the `[ system ]` and `[ molecules ]` sections. Then we minimise the energy of the lipids alone, to remove any overlap between beads placed too close.

In [ ]:
!cp ../martini_lipids.top topol.top
!printf "\n[ system ]\nDPPC self-assembly\n\n[ molecules ]\nDPPC 128\n" >> topol.top
!tail -8 topol.top
!gmx grompp -f ../martini_em.mdp -c 128_noW.gro -p topol.top -o min_noW.tpr
!gmx mdrun -deffnm min_noW -c 128_minimized.gro

Then we use `solvate` to add 6 water beads per lipid (that corresponds to 24 water molecules per lipid), taken from an equilibrated box of Martini water (`water.gro`), and minimise again. `solvate` also adds the number of water beads to the topology.

In [ ]:
!gmx solvate -cp 128_minimized.gro -cs ../water.gro -o waterbox.gro -maxsol 768 -radius 0.21 -p topol.top
!gmx grompp -f ../martini_em.mdp -c waterbox.gro -p topol.top -o min.tpr
!gmx mdrun -deffnm min -c minimized.gro

Now the self-assembly run: 30 ns at 323 K, a temperature at which a DPPC membrane is in the fluid phase, with isotropic pressure coupling (the box can change its volume but not its shape). It takes a few minutes on a GPU (about half an hour on two CPU cores); the bilayer usually forms within 10-20 ns.

In [ ]:
!gmx grompp -f ../martini_selfassembly.mdp -c minimized.gro -p topol.top -o selfassembly.tpr
!gmx mdrun -deffnm selfassembly -v

Molecules cross the sides of the periodic box: to look at the trajectory we first make them whole (`-pbc mol`). Then let's see what happened:

In [ ]:
!echo System | gmx trjconv -s selfassembly.tpr -f selfassembly.xtc -pbc mol -o selfassembly_whole.xtc
show_cg('selfassembly.tpr', 'selfassembly_whole.xtc')

Did you get a bilayer? It can take a few tens of nanoseconds; if needed, continue the simulation (`gmx convert-tpr -s selfassembly.tpr -extend 20000 -o selfassembly.tpr` and then `gmx mdrun -deffnm selfassembly -cpi selfassembly.cpt -v`). Sometimes the lipids form a bilayer with a pore or some other defect: you can try again with a different random starting configuration.

To quantify the formation of the bilayer we follow the orientation of the lipids: for each lipid we take the unit vector from the phosphate to the end of a tail and compute, for each axis, the average of the square of its component. For randomly oriented lipids the three values are 1/3; in a bilayer the value along the normal to the membrane approaches 1. We also print the normal to the bilayer at the end.

In [ ]:
def lipid_orientation(u, head='PO4', tail='C4A'):
    """For each frame, <n_x^2>, <n_y^2>, <n_z^2> of the unit vectors from the head bead to the tail end of each lipid."""
    heads, tails = u.select_atoms(f'name {head}'), u.select_atoms(f'name {tail}')
    times, values = [], []
    for ts in u.trajectory:
        d = tails.positions - heads.positions
        d -= ts.dimensions[:3] * np.round(d / ts.dimensions[:3])        # nearest periodic image
        d /= np.linalg.norm(d, axis=1)[:, None]
        times.append(ts.time / 1000)
        values.append((d ** 2).mean(axis=0))
    return np.array(times), np.array(values)

u = mda.Universe('selfassembly.tpr', 'selfassembly.xtc')
t, n2 = lipid_orientation(u)
for i, axis in enumerate('xyz'):
    plt.plot(t, n2[:, i], label=f'<n$_{axis}^2$>')
plt.axhline(1 / 3, color='k', ls=':')
plt.xlabel('Time (ns)')
plt.ylabel('Orientation of the lipids')
plt.legend()
plt.show()
normal = 'xyz'[np.argmax(n2[-10:].mean(axis=0))]
print('the normal to the membrane is along', normal)

For the analysis it is convenient to have the bilayer in the $xy$ plane, with the normal along $z$. If the normal is along $x$ or $y$ we rotate the system:

In [ ]:
rotation = {'x': '0 90 0', 'y': '90 0 0', 'z': '0 0 0'}[normal]
!gmx editconf -f selfassembly.gro -rotate {rotation} -o selfassembly_xy.gro

The self-assembly was done with isotropic pressure coupling: the bilayer is probably under tension. We need a simulation in which the area of the bilayer can reach its equilibrium value, with independent pressure coupling in the plane of the membrane and along its normal (`pcoupltype = semiisotropic`). At the same pressure in the two directions the membrane is at zero surface tension. We first run 1 ns with a shorter time step (10 fs), because the box after the rotation is not equilibrated, and then 20 ns.

In [ ]:
!gmx grompp -f ../martini_eq.mdp -c selfassembly_xy.gro -p topol.top -o eq.tpr
!gmx mdrun -deffnm eq -v
!gmx grompp -f ../martini_md.mdp -c eq.gro -t eq.cpt -p topol.top -o md.tpr
!gmx mdrun -deffnm md -v

The (projected) area per lipid is the area of the box in the plane of the membrane divided by the number of lipids in one leaflet. Here we compare it during the self-assembly (where the bilayer may be in any plane) and the following simulation.

In [ ]:
def box_area(edr, plane='xy'):
    """Time (ns) and area (nm^2) of the box face in the plane, from an .edr file."""
    a, b = plane
    !printf "Box-{a.upper()}\nBox-{b.upper()}\n\n" | gmx energy -f {edr} -o box.xvg > /dev/null 2>&1
    data, _ = read_xvg('box.xvg')
    return data[:, 0] / 1000, data[:, 1] * data[:, 2]

plane = {'x': 'yz', 'y': 'xz', 'z': 'xy'}[normal]
t1, area1 = box_area('selfassembly.edr', plane)
t2, area2 = box_area('md.edr', 'xy')
plt.plot(t1, area1 / 64, label='self-assembly (isotropic)')
plt.plot(t1[-1] + 1 + t2, area2 / 64, label='semi-isotropic')
plt.xlabel('Time (ns)')
plt.ylabel('Area per lipid (nm$^2$)')
plt.legend()
plt.show()
print(f'area per lipid in the last 10 ns: {np.mean(area2[t2 > t2[-1] - 10]) / 64:.3f} nm^2')

## Building a bilayer with insane

Self-assembly is spectacular, but it is slow and not always successful, especially for mixtures of lipids and for large systems. Bilayers are usually built with a program, here `insane` (INSert membrANE, Wassenaar et al., JCTC 2015), that places the lipids on a grid in two leaflets and adds water and ions. We use the version distributed with the Martini 3 lipid parameters, which knows their structures.

Some options:

*   `-l DPPC:1` lipid types and their relative numbers in the leaflets (e.g. `-l DPPC:7 -l CHOL:3` for 30% cholesterol); `-u` can be used to give a different composition to the upper leaflet
*   `-x -y -z` size of the box (nm)
*   `-sol W` solvent, `-salt 0.15` NaCl concentration (M)
*   `-o` output structure, `-p` output topology (it includes `martini.itp`: we replace it with our list of force-field files)

We use a box 12 nm high, so that there are more than 20 water beads per lipid. With much less water, the thin layer of Martini water between the periodic images of the membrane tends to freeze below about 315 K: a known artefact of the model, that you should always check for (see the diffusion below).

In [ ]:
%cd $martini_path
%mkdir -p dppc
%cd dppc
!cp ../*.itp .
!python3 $MARTINI_DIR/insane.py -l DPPC:1 -x 6.5 -y 6.5 -z 12 -sol W -salt 0.15 -o bilayer.gro -p insane.top 2>&1 | tail -6
# topology: our force-field files and the [ system ] and [ molecules ] sections written by insane
!cp ../martini_lipids.top topol.top
!sed -n '/\[ system \]/,$p' insane.top >> topol.top
!tail -6 topol.top
show_cg('bilayer.gro')

**Question:** how many lipids are in each leaflet? How many water beads per lipid (and how many water molecules do they represent), and how many ions?

Then the usual sequence: minimisation, a short equilibration with a shorter time step, and the production run, 20 ns at 323 K.

In [ ]:
!gmx grompp -f ../martini_em.mdp -c bilayer.gro -p topol.top -o min.tpr
!gmx mdrun -deffnm min
!gmx grompp -f ../martini_eq.mdp -c min.gro -p topol.top -o eq.tpr
!gmx mdrun -deffnm eq -v
!gmx grompp -f ../martini_md.mdp -c eq.gro -t eq.cpt -p topol.top -o md.tpr
!gmx mdrun -deffnm md -v

## Analysis of a bilayer

We compute some of the properties that are measured experimentally and used to validate (and to parameterise) lipid force fields:

*   the **area per lipid** $A_L$, from the box;
*   the **thickness**, here the average distance $D_{PP}$ between the phosphate beads of the two leaflets (in experiments the closely related head-to-head distance $D_{HH}$ is measured by X-ray scattering);
*   the **order parameter** of the tails, $P_2 = \frac{1}{2}\langle 3\cos^2\theta - 1\rangle$, where $\theta$ is the angle between a bond and the membrane normal ($z$). $P_2=1$ for bonds parallel to the normal, $0$ for random orientations and $-0.5$ for bonds parallel to the membrane plane. It is the coarse-grained equivalent of the order parameters $S_{CD}$ measured by NMR;
*   the **density profile** of the different groups along the normal;
*   the **lateral diffusion coefficient** of the lipids.

The function below computes the first three for each frame. Each lipid is assigned to a leaflet from the position of its phosphate with respect to the centre of the membrane.

In [ ]:
TAILS = ['GL1 C1A C2A C3A C4A', 'GL2 C1B C2B C3B C4B']    # beads along the two DPPC tails

def leaflet_side(heads, box):
    """+1 for the head beads above the centre of the membrane, -1 for those below (periodic along z)."""
    z, lz = heads.positions[:, 2], box[2]
    angle = 2 * np.pi * z / lz                      # the centre is a circular average: it works also if the membrane crosses the box
    center = lz * np.arctan2(np.sin(angle).mean(), np.cos(angle).mean()) / (2 * np.pi)
    dz = (z - center + lz / 2) % lz - lz / 2
    return np.sign(dz), dz

def bilayer_properties(u, lipid='DPPC', tails=TAILS, start_ns=0):
    """Per frame: time (ns), area per lipid (nm^2, box area / phospholipids per leaflet), P-P thickness (nm),
    and the order parameter P2 of each bond along the tails (array frames x bonds)."""
    heads = u.select_atoms(f'resname {lipid} and name PO4')
    bonds = [(u.select_atoms(f'resname {lipid} and name {a}'), u.select_atoms(f'resname {lipid} and name {b}'))
             for tail in tails for a, b in zip(tail.split()[:-1], tail.split()[1:])]
    t, area, thickness, p2 = [], [], [], []
    for ts in u.trajectory:
        if ts.time / 1000 < start_ns:
            continue
        side, dz = leaflet_side(heads, ts.dimensions)
        t.append(ts.time / 1000)
        area.append(ts.dimensions[0] * ts.dimensions[1] / 100 / (len(heads) / 2))
        thickness.append((dz[side > 0].mean() - dz[side < 0].mean()) / 10)
        values = []
        for a, b in bonds:
            d = b.positions - a.positions
            d -= ts.dimensions[:3] * np.round(d / ts.dimensions[:3])
            cos2 = (d[:, 2] / np.linalg.norm(d, axis=1)) ** 2
            values.append(0.5 * (3 * cos2.mean() - 1))
        p2.append(values)
    return np.array(t), np.array(area), np.array(thickness), np.array(p2)

u = mda.Universe('md.tpr', 'md.xtc')
t, area, thickness, p2 = bilayer_properties(u)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(t, area)
axes[0].set_ylabel('Area per lipid (nm$^2$)')
axes[1].plot(t, thickness)
axes[1].set_ylabel('$D_{PP}$ (nm)')
axes[2].plot(t, p2.mean(axis=1))
axes[2].set_ylabel('Average $P_2$')
for ax in axes:
    ax.set_xlabel('Time (ns)')
plt.tight_layout()
plt.show()

skip = t > 5          # the first 5 ns are excluded from the averages
print(f'area per lipid: {area[skip].mean():.3f} ± {area[skip].std():.3f} nm^2')
print(f'thickness D_PP: {thickness[skip].mean():.2f} ± {thickness[skip].std():.2f} nm')
labels = [f'{a}-{b}' for tail in TAILS for a, b in zip(tail.split()[:-1], tail.split()[1:])]
plt.figure(figsize=(8, 4))
plt.plot(labels, p2[skip].mean(axis=0), 'o-')
plt.ylabel('$P_2$')
plt.xticks(rotation=45)
plt.show()

The fluctuations shown after the ± are those of single frames: they are not the uncertainty of the average, which is smaller (but frames close in time are correlated: think about how you could estimate it, e.g. with block averages).

**Density profiles.** `gmx density` computes the number density of groups of beads along the normal, with the centre of the bilayer at zero (`-center`). We first make an index file with the groups.

In [ ]:
!printf "a NC3\na PO4\na GL1 GL2\na C1A C2A C3A C4A C1B C2B C3B C4B\nq\n" | gmx make_ndx -f md.tpr -o groups.ndx > /dev/null
!grep '\[' groups.ndx
groups = {'NC3': 'choline', 'PO4': 'phosphate', 'GL1_GL2': 'glycerol', 'C1A_C2A_C3A_C4A_C1B_C2B_C3B_C4B': 'tails', 'W': 'water', 'ION': 'ions'}
plt.figure(figsize=(8, 5))
for g, label in groups.items():
    # the first group (DPPC) is used to centre the bilayer, the second is the one whose density is computed
    !rm -f density.xvg
    !printf "DPPC\n{g}\n" | gmx density -f md.xtc -s md.tpr -n groups.ndx -dens number -center -sl 100 -b 5000 -o density.xvg > /dev/null 2>&1
    data, _ = read_xvg('density.xvg')
    plt.plot(data[:, 0], data[:, 1], label=label)
plt.xlabel('z (nm)')
plt.ylabel('Number density (beads/nm$^3$)')
plt.legend()
plt.show()

**Lateral diffusion.** The mean square displacement (MSD) of the phosphate beads in the plane of the membrane grows linearly with time, $\langle |\Delta \mathbf{r}_{xy}(\tau)|^2 \rangle = 4 D \tau$. `gmx msd` computes it and fits $D$; we first remove the jumps of the beads across the sides of the box (`-pbc nojump`).

In [ ]:
!echo System | gmx trjconv -s md.tpr -f md.xtc -pbc nojump -o md_nojump.xtc > /dev/null 2>&1
# -trestart: a new time origin every 500 ps; -beginfit and -endfit: the interval of tau (ps) used for the linear fit
!gmx msd -s md.tpr -f md_nojump.xtc -sel "name PO4" -lateral z -trestart 500 -beginfit 1000 -endfit 10000 -o msd.xvg > msd.log 2>&1
data, legends = read_xvg('msd.xvg')
print(legends[0])          # the fitted diffusion coefficient
plt.plot(data[:, 0] / 1000, data[:, 1])
# the same for water: if the diffusion coefficient is close to zero, the water has frozen
!gmx msd -s md.tpr -f md_nojump.xtc -sel "name W" -trestart 500 -beginfit 1000 -endfit 10000 -o msd_water.xvg > msd_water.log 2>&1
print(read_xvg('msd_water.xvg')[1][0])
plt.xlabel('$\\tau$ (ns)')
plt.ylabel('MSD$_{xy}$ (nm$^2$)')
plt.show()

**Comparison with experiments.** The Martini developers compared their lipids with experimental data; the experimental values and those of the published Martini 3 simulations are in the folder `Data/martini/benchmark` (Pedersen et al., 2025). Do your results agree with those of the published simulations? And with the experiments?

In [ ]:
import pandas as pd
for quantity, name in [('apl', 'area per lipid (nm^2)'), ('dhh', 'D_HH (nm)')]:
    table = pd.read_csv(f'{MARTINI_DIR}/benchmark/figure1_{quantity}_data.csv')
    print(name)
    print(table[table.lipid.isin(['DPPC', 'POPC', 'DOPC', 'DMPC'])].to_string(index=False), '\n')

**Questions:** What is the area per lipid and the thickness of your bilayer? How do they compare with the experiments? Which parts of the lipids are more ordered? The diffusion coefficient of lipids in a fluid membrane is of the order of $10^{-7}$ cm$^2$/s ($10$ μm$^2$/s): is the diffusion in Martini faster or slower? Why do you expect the dynamics of a coarse-grained model to be faster than the real one?

**To visualise the simulations** with VMD or ChimeraX you need a file that tells them which beads are bonded. MDAnalysis reads the bonds from the `.tpr` file and writes them in a PDB file (CONECT records). Open `md_whole.pdb` and then load `md_whole.xtc` into it; in VMD use, e.g., the `VDW` or `CPK` drawing method.

In [ ]:
!echo System | gmx trjconv -s md.tpr -f md.xtc -pbc mol -o md_whole.xtc > /dev/null 2>&1
u = mda.Universe('md.tpr', 'md_whole.xtc')
u.select_atoms('all').write('md_whole.pdb', bonds='conect')

## References

*   Laio A, Parrinello M (2002) Escaping free-energy minima. *PNAS* 99:12562
*   Barducci A, Bussi G, Parrinello M (2008) Well-tempered metadynamics: a smoothly converging and tunable free-energy method. *Phys Rev Lett* 100:020603
*   Tiwary P, Parrinello M (2015) A time-independent free energy estimator for metadynamics. *J Phys Chem B* 119:736
*   The PLUMED consortium (2019) Promoting transparency and reproducibility in enhanced molecular simulations. *Nat Methods* 16:670
*   Marrink SJ, de Vries AH, Mark AE (2004) Coarse grained model for semiquantitative lipid simulations. *J Phys Chem B* 108:750
*   Souza PCT et al. (2021) Martini 3: a general purpose force field for coarse-grained molecular dynamics. *Nat Methods* 18:382
*   Pedersen KB et al. (2025) The Martini 3 lipidome: expanded and refined parameters improve lipid phase behavior. *ACS Cent Sci*, https://doi.org/10.1021/acscentsci.5c00755
*   Borges-Araújo L et al. (2023) Martini 3 coarse-grained force field for cholesterol. *J Chem Theory Comput* 19:7387
*   Wassenaar TA et al. (2015) Computational lipidomics with insane: a versatile tool for generating custom membranes for molecular simulations. *J Chem Theory Comput* 11:2144